In [5]:
# 下载Deepresearch-9k 数据集作为通用训练数据


from datasets import load_dataset
import os

# 1. 指定下载与缓存位置
# 使用 cache_dir 参数可以将数据下载到你指定的硬盘路径，避免占用系统盘空间
custom_cache_dir = "./my_dataset_cache" 
os.makedirs(custom_cache_dir, exist_ok=True)

print("正在加载数据集并下载至:", custom_cache_dir)
ds = load_dataset("artillerywu/DeepResearch-9K", cache_dir=custom_cache_dir)

# 2. 指定下载/保留的字段
# 根据论文 [cite: 90]，该数据集包含 'question', 'difficulty', 'search_trajectory', 'final_answer'
# 我们使用 remove_columns 方法舍弃不需要的字段
cols_to_remove = ["search trajectory"]
ds_filtered = ds.remove_columns(cols_to_remove)

# 3. 转为 CSV 保存
# 因为数据集通常包含多个划分（如 train, test），建议遍历保存
for split, dataset in ds_filtered.items():
    output_name = f"deep_research_9k_{split}.csv"
    dataset.to_csv(output_name, index=False) # index=False 不保存行索引
    print(f"已将 {split} 部分保存至: {output_name}")

正在加载数据集并下载至: ./my_dataset_cache


Creating CSV from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

已将 train 部分保存至: deep_research_9k_train.csv


In [6]:
# 分析 DeepResearch-hard

import pandas as pd

df = pd.read_csv("general_dataset/deep_research_hard.csv")

print(df.difficulty.value_counts())
print(df.difficulty.value_counts(normalize=True))

difficulty
3    2288
2     860
1     826
Name: count, dtype: int64
difficulty
3    0.575742
2    0.216407
1    0.207851
Name: proportion, dtype: float64


In [14]:
# 从 DeepResearch-9k 的 hard 数据集 抽取出 SFT+RL 实际使用的数据集

import pandas as pd
from pathlib import Path

# 加载完整数据集
df = pd.read_csv("general_dataset/deep_research_hard.csv")

# 打印出当前的列名，帮你排查到底叫什么
print("📌 原始数据包含的列名有：", df.columns.tolist())

# 按照 difficulty 字段进行分层抽样，总数取 1000
sampled_df = df.groupby('difficulty', group_keys=False).sample(frac=0.25163563160544, random_state=42)

print("📌 sampled_df数据包含的列名有：", sampled_df.columns.tolist())

# 将抽样后的数据分为两部分：
# 1. 潜力 SFT 库 (用于跑 Agent 框架获取轨迹)
sft_candidates = sampled_df.sample(n=600, random_state=42) 
# 2. 预留 GRPO 验证/训练库 (不运行 Agent，仅保留问答对)
grpo_pool = sampled_df.drop(sft_candidates.index)

if 'difficulty' not in sft_candidates.columns:
    # 尝试将 index 重置为普通列
    sft_candidates = sft_candidates.reset_index()
    grpo_pool = grpo_pool.reset_index()

# 打印出当前的列名，帮你排查到底叫什么
print("📌 当前数据包含的列名有：", sft_candidates.columns.tolist())

columns_to_keep = ['question', 'difficulty', 'final answer'] 

# 过滤数据框，只保留指定列
sft_final = sft_candidates[columns_to_keep]
grpo_final = grpo_pool[columns_to_keep]


# 保存数据
output_dir = Path("general_dataset")
output_dir.mkdir(parents=True, exist_ok=True)

# 2. 保存为 CSV 
# 增加 encoding="utf-8-sig"，防止在 Windows Excel 中打开时中文乱码
sft_final.to_csv(output_dir / "sft_candidates.csv", index=False, encoding="utf-8-sig")
grpo_final.to_csv(output_dir / "grpo_pool.csv", index=False, encoding="utf-8-sig")

# 3. 保存为 JSON
# sft_candidates.to_json(output_dir / "sft_candidates.json", orient="records", force_ascii=False, indent=2)
# grpo_pool.to_json(output_dir / "grpo_pool.json", orient="records", force_ascii=False, indent=2)

# 【进阶推荐】：对于 LLM 训练数据 (SFT/RL)，业界通常更推荐使用 JSONL (JSON Lines) 格式
# 如果你的 Agent 框架或训练框架支持，建议使用以下代码替代上面的 JSON 保存：
sft_final.to_json(output_dir / "sft_candidates.jsonl", orient="records", lines=True, force_ascii=False)
grpo_final.to_json(output_dir / "grpo_pool.jsonl", orient="records", lines=True, force_ascii=False)

print(f"✅ 数据成功保存至 {output_dir.absolute()} 目录下！")

📌 原始数据包含的列名有： ['question', 'difficulty', 'final answer']
📌 sampled_df数据包含的列名有： ['question', 'difficulty', 'final answer']
📌 当前数据包含的列名有： ['question', 'difficulty', 'final answer']
✅ 数据成功保存至 D:\DissertationProject\DissertationProject\dataset\general_dataset 目录下！


In [4]:
# 清理 CS QA数据


import os
import json
from pathlib import Path
import pandas as pd

dir = "D:\\DissertationProject\\DissertationProject\\dataset\\CS_dataset"


with open(os.path.join(dir, "cs_qa_original.jsonl"), "r", encoding="utf-8") as f:
    cs_qa_pairs = [json.loads(line) for line in f]
    
print(f"共找到 {len(cs_qa_pairs)} 条数据")

output_dir = Path("CS_dataset")
output_dir.mkdir(parents=True, exist_ok=True)

cs_qa_pairs_df = pd.DataFrame(cs_qa_pairs)


# 去除所有 "answer":"无法确定" 的项
cs_qa_pairs_df_cleaned = cs_qa_pairs_df[cs_qa_pairs_df["answer"] != "无法确定"]

# 只保留 question, answer, entity(改名为 seed_entity)
cs_qa_pairs_df_cleaned = cs_qa_pairs_df_cleaned[['question', 'answer', 'entity']].rename(columns={'entity': 'seed_entity'})
cs_qa_pairs_df_cleaned.to_csv(output_dir / "cs_qa_cleaned.csv", index=False, encoding="utf-8-sig")

# 3. 保存为 JSON
# sft_candidates.to_json(output_dir / "sft_candidates.json", orient="records", force_ascii=False, indent=2)
# grpo_pool.to_json(output_dir / "grpo_pool.json", orient="records", force_ascii=False, indent=2)

# 【进阶推荐】：对于 LLM 训练数据 (SFT/RL)，业界通常更推荐使用 JSONL (JSON Lines) 格式
# 如果你的 Agent 框架或训练框架支持，建议使用以下代码替代上面的 JSON 保存：
cs_qa_pairs_df_cleaned.to_json(output_dir / "cs_qa_cleaned.jsonl", orient="records", lines=True, force_ascii=False)

print(f"清理后剩余 {len(cs_qa_pairs_df_cleaned)} 条数据")

print(f"✅ 数据成功保存至 {output_dir.absolute()} 目录下！")


共找到 436 条数据
清理后剩余 311 条数据
✅ 数据成功保存至 D:\DissertationProject\DissertationProject\dataset\CS_dataset 目录下！


In [5]:
# 拆分出 SFT / RL / Benchmark 数据集


import os
import json
from pathlib import Path
import pandas as pd

# 设定路径 (使用 raw string 避免 Windows 路径转义问题)
dir = r"D:\DissertationProject\DissertationProject\dataset\CS_dataset"
output_dir = Path(dir) / "splits"
output_dir.mkdir(parents=True, exist_ok=True)

# 读取数据
csv_path = os.path.join(dir, "cs_qa_cleaned.csv")
all_cs_qa_pairs_df = pd.read_csv(csv_path)
total_count = len(all_cs_qa_pairs_df)
print(f"共找到 {total_count} 条数据\n")

# 1. 随机打乱数据 (设定 random_state=42 保证每次划分结果一致，确保实验可复现)
shuffled_df = all_cs_qa_pairs_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 2. 执行硬切分 (Hard Split)
# Benchmark: 50 条绝对隔离
benchmark_df = shuffled_df.iloc[:50]

# SFT Seeds: 100 条 (后续由强大模型跑 IterResearch 展开为 Step-level MDP 轨迹)
sft_seeds_df = shuffled_df.iloc[50:150]

# RL Exclusive: 剩余的 161 条 (针对 311 条总数的情况)
rl_exclusive_df = shuffled_df.iloc[150:]

# 3. 组装 RL 训练集 (RL Exclusive + SFT Seeds)
# 在 GRPO/GSPO 等对齐阶段，复用 SFT 的 数据 能够巩固模型对工具调用语法格式的记忆
rl_qa_df = pd.concat([rl_exclusive_df, sft_seeds_df]).reset_index(drop=True)

# 4. 导出为 CSV 和 JSONL
def save_splits(df, filename_prefix):
    csv_out = output_dir / f"{filename_prefix}.csv"
    jsonl_out = output_dir / f"{filename_prefix}.jsonl"
    
    df.to_csv(csv_out, index=False)
    # orient="records", lines=True 是标准 JSONL 格式
    df.to_json(jsonl_out, orient="records", lines=True, force_ascii=False)

save_splits(benchmark_df, f"benchmark_{len(benchmark_df)}")
save_splits(sft_seeds_df, f"sft_seeds_{len(sft_seeds_df)}")
save_splits(rl_qa_df, f"rl_qa_{len(rl_qa_df)}")

# 打印执行摘要
print("="*40)
print("数据拆分完成！各数据集规模如下：")
print("="*40)
print(f"- Benchmark 测试集: {len(benchmark_df)} 条 (绝对隔离)")
print(f"- SFT 种子集:      {len(sft_seeds_df)} 条 (待输入强大模型展开为 MDP 轨迹)")
print(f"- RL QA集:     {len(rl_qa_df)} 条 (包含 {len(rl_exclusive_df)} 条专属 + {len(sft_seeds_df)} 条 SFT 复用)")
print("="*40)
print(f"文件已保存至: {output_dir}")

共找到 311 条数据

数据拆分完成！各数据集规模如下：
- Benchmark 测试集: 50 条 (绝对隔离)
- SFT 种子集:      100 条 (待输入强大模型展开为 MDP 轨迹)
- RL QA集:     261 条 (包含 161 条专属 + 100 条 SFT 复用)
文件已保存至: D:\DissertationProject\DissertationProject\dataset\CS_dataset\splits
